
# FAD-RNet — Fabric Anomaly Detection via Reverse Distillation
### (WideResNet-50 Teacher + OCBE + Frequency-Domain Student Decoder)

This notebook implements the model exactly as specified in your architecture diagram:

```
Fabric Image → Preprocess (resize 256×256, normalize)
             → Frozen Teacher Encoder (WideResNet-50, ImageNet)
             → multi-scale features F1, F2, F3, F4
             → OCBE (Multi-scale Feature Fusion + Channel-Spatial Attention)
             → Bottleneck Feature
             → Student Decoder (Reverse Distillation)
             → FDFM Stage 1 → FDFM Stage 2 (high-freq + low-freq + residual)
             → Reconstructed Teacher Features
             → Teacher ↔ Student difference → Anomaly Map / Score
             → Gaussian Smoothing → Image Score & Pixel Map
             → Normal/Defect decision + Localization
```

**How training works (important):** this is an *unsupervised, reconstruction-based* method.
The student decoder is trained **only on normal ("good") fabric images** — it learns to
reconstruct the teacher's features for normal texture. At test time, on a defective image
the student **fails** to reconstruct the teacher's features around the defect, producing a
large teacher–student discrepancy exactly at that location → the anomaly map.

**About your dataset — read this carefully:**
Your dataset has `train/good` and `test/{good, holes, lines, stains}` — there are **no
pixel-level ground-truth masks**. That means:
- ✅ We *can* compute full **image-level** metrics: AUROC, F1, confusion matrix, ROC/PR curves,
  because the test folder names (`good` vs `holes/lines/stains`) act as image-level labels.
- ❌ We *cannot* compute pixel-level metrics (pixel-AUROC, PRO score, IoU) because there is no
  ground-truth defect mask to compare the anomaly map against. The anomaly maps / localization
  heatmaps are still produced and shown — they're just evaluated **qualitatively** (visually),
  not with a numeric pixel score. If you later get masks, see the note at the very end of this
  notebook for the 5 lines needed to add pixel-AUROC.

Every place you are likely to need to edit is marked with a cell titled **`# >>> EDIT`**.


## 1. Setup — install & imports

In [ ]:

# On Kaggle, torch/torchvision/scipy/sklearn/matplotlib are pre-installed.
# This cell is safe to run as-is; it only installs anything that's missing.
import sys, subprocess

def _ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for _p in ["torch", "torchvision", "scipy", "scikit-learn", "matplotlib", "seaborn", "tqdm"]:
    _ensure(_p if _p != "scikit-learn" else "sklearn")

import os, glob, random, json, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                              auc, confusion_matrix, classification_report,
                              f1_score, ConfusionMatrixDisplay)
from tqdm.auto import tqdm

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())



## 2. Config — **>>> EDIT THIS CELL FIRST**

Based on the dataset structure you showed me:

```
dataset/
├── train/
│   └── good/
└── test/
    ├── good/
    ├── holes/
    ├── lines/
    └── stains/
```

Add this dataset to your Kaggle notebook (`+ Add Input`), then it will be mounted under
`/kaggle/input/<dataset-slug>/...`. **Change `DATA_ROOT` below to the exact path** — the
auto-detect logic will try to find it for you and print what it found, but double-check the
printed path is correct before continuing.


In [ ]:

# >>> EDIT: if auto-detection below doesn't find the right folder, hardcode it here, e.g.:
# DATA_ROOT = "/kaggle/input/dataset-for-fabric-anomaly-detection-for-efficientnetb0/dataset"
DATA_ROOT = None   # leave as None to auto-detect

if DATA_ROOT is None:
    candidates = glob.glob("/kaggle/input/**/train/good", recursive=True)
    if len(candidates) == 0:
        # fall back for local/non-Kaggle testing
        candidates = glob.glob("./**/train/good", recursive=True)
    assert len(candidates) > 0, (
        "Could not auto-detect the dataset. Set DATA_ROOT manually to the folder "
        "that CONTAINS 'train' and 'test' subfolders."
    )
    DATA_ROOT = os.path.dirname(os.path.dirname(candidates[0]))

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
print("DATA_ROOT :", DATA_ROOT)
print("TRAIN_DIR :", TRAIN_DIR, "->", os.listdir(TRAIN_DIR))
print("TEST_DIR  :", TEST_DIR,  "->", os.listdir(TEST_DIR))

# >>> EDIT: the normal-class folder name in both train/ and test/ (default 'good')
NORMAL_CLASS_NAME = "good"

# >>> EDIT: hyperparameters
IMG_SIZE     = 256          # diagram specifies 256x256
BATCH_SIZE   = 16           # lower to 8 if you hit CUDA out-of-memory
NUM_EPOCHS   = 100          # fabric texture reconstruction usually needs 50-150 epochs
LR           = 1e-3
WEIGHT_DECAY = 1e-5
VAL_SPLIT    = 0.1          # fraction of train/good held out for validation / checkpointing
SEED         = 42
GAUSS_SIGMA  = 4            # smoothing on the final anomaly map
NUM_WORKERS  = 2
OUTPUT_DIR   = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)


## 3. Dataset & Transforms

In [ ]:

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),     # fabric texture is usually orientation-invariant
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

class FlatImageFolder(Dataset):
    """All images directly inside `root`, single implicit label."""
    def __init__(self, root, transform):
        self.paths = sorted([p for p in glob.glob(os.path.join(root, "*"))
                              if p.lower().endswith(IMG_EXTS)])
        self.transform = transform
        assert len(self.paths) > 0, f"No images found in {root}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), path


class LabeledTestDataset(Dataset):
    """Walks TEST_DIR/<class_name>/*.jpg ; label 0 = normal, 1 = defect."""
    def __init__(self, root, normal_class_name, transform):
        self.transform = transform
        self.samples = []       # (path, binary_label, class_name)
        class_names = sorted(os.listdir(root))
        for cname in class_names:
            cdir = os.path.join(root, cname)
            if not os.path.isdir(cdir):
                continue
            label = 0 if cname == normal_class_name else 1
            for p in glob.glob(os.path.join(cdir, "*")):
                if p.lower().endswith(IMG_EXTS):
                    self.samples.append((p, label, cname))
        assert len(self.samples) > 0, f"No images found under {root}"

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, cname = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label, cname, path


# ---- build datasets ----
full_train_ds = FlatImageFolder(os.path.join(TRAIN_DIR, NORMAL_CLASS_NAME), train_transform)
n_val = max(1, int(len(full_train_ds) * VAL_SPLIT))
n_train = len(full_train_ds) - n_val
train_ds, val_ds = random_split(full_train_ds, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(SEED))
# random_split shares the SAME underlying dataset/transform for both subsets, but
# validation must use eval_transform (no augmentation) -> rebuild both subsets
# explicitly from their file paths with the correct transform each.
val_paths = [full_train_ds.paths[i] for i in val_ds.indices]
train_paths = [full_train_ds.paths[i] for i in train_ds.indices]

class PathSubset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths; self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), self.paths[idx]

train_ds = PathSubset(train_paths, train_transform)
val_ds   = PathSubset(val_paths, eval_transform)
test_ds  = LabeledTestDataset(TEST_DIR, NORMAL_CLASS_NAME, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print(f"train (good, aug)   : {len(train_ds)} images")
print(f"val   (good, held out): {len(val_ds)} images")
print(f"test  (all classes) : {len(test_ds)} images")
from collections import Counter
print("test class breakdown:", Counter([c for _, _, c in test_ds.samples]))


### 3.1 Sanity check — visualize a few training images

In [ ]:

def denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (t * std + mean).clamp(0,1)

fig, axes = plt.subplots(1, 5, figsize=(15,3))
for i in range(5):
    img, _ = train_ds[i]
    axes[i].imshow(denorm(img).permute(1,2,0).numpy())
    axes[i].axis("off")
fig.suptitle("Sample training images (normal fabric only)")
plt.tight_layout(); plt.show()



## 4. Model — FAD-RNet

Implements every block in the diagram:
- **TeacherEncoder** — frozen WideResNet-50 (ImageNet pretrained), returns `F1..F4`.
- **OCBE** — Multi-scale Feature Fusion (`MFF`, one projection conv per scale, all resampled
  to `F4`'s resolution) + Channel-Spatial Attention (`CSA`, squeeze-excite channel gate +
  spatial gate) → produces the single **Bottleneck Feature**.
- **StudentDecoder** — 3 transposed-conv upsampling blocks (reverse of the teacher's
  layer4→layer3→layer2→layer1), mirroring "Reverse Distillation".
- **FDFM** (×2, after the first two decoder blocks) — splits the feature map into
  low-/high-frequency bands via FFT + a Gaussian frequency mask, processes each band with its
  own conv, fuses them, and adds a residual connection. Fabric defects (holes / lines / stains)
  live mostly in the high-frequency band, so this lets the decoder pay targeted attention to them.


In [ ]:

# ---------------------------------------------------------------------
# 4.1 Frozen Teacher Encoder
# ---------------------------------------------------------------------
class TeacherEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = Wide_ResNet50_2_Weights.IMAGENET1K_V2 if pretrained else None
        net = wide_resnet50_2(weights=weights)
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1 = net.layer1   # F1: 256 ch,  1/4
        self.layer2 = net.layer2   # F2: 512 ch,  1/8
        self.layer3 = net.layer3   # F3: 1024 ch, 1/16
        self.layer4 = net.layer4   # F4: 2048 ch, 1/32
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    @torch.no_grad()
    def forward(self, x):
        x = self.stem(x)
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [f1, f2, f3, f4]


# ---------------------------------------------------------------------
# 4.2 OCBE = MFF (multi-scale feature fusion) + CSA (channel-spatial attention)
# ---------------------------------------------------------------------
class ChannelSpatialAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
        )
        self.spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        ca = torch.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))
        x = x * ca
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.sigmoid(self.spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * sa


class OCBE(nn.Module):
    """Fuses F1..F4 (resampled to F4's resolution) then applies CSA.
    Output resolution = F4's resolution (H/32), which is exactly where the
    student decoder needs to start from to reconstruct F3 -> F2 -> F1."""
    def __init__(self, in_channels=(256, 512, 1024, 2048), out_channels=2048):
        super().__init__()
        mid = out_channels // 4
        self.proj = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, mid, kernel_size=3, stride=2 ** (3 - i), padding=1, bias=False)
                if i < 3 else nn.Conv2d(c, mid, kernel_size=1, bias=False),
                nn.BatchNorm2d(mid),
                nn.ReLU(inplace=True),
            ) for i, c in enumerate(in_channels)
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(mid * 4, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.csa = ChannelSpatialAttention(out_channels)

    def forward(self, feats):
        target_size = feats[-1].shape[-2:]
        projected = []
        for p, f in zip(self.proj, feats):
            f = p(f)
            if f.shape[-2:] != target_size:
                f = F.adaptive_avg_pool2d(f, target_size)
            projected.append(f)
        fused = self.fuse(torch.cat(projected, dim=1))
        return self.csa(fused)


# ---------------------------------------------------------------------
# 4.3 FDFM — Frequency-Domain Fusion Module (high + low freq + residual)
# ---------------------------------------------------------------------
class FDFM(nn.Module):
    def __init__(self, channels, radius_ratio=0.25):
        super().__init__()
        self.radius_ratio = radius_ratio
        self.low_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True))
        self.high_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False),
            nn.BatchNorm2d(channels))
        self.act = nn.ReLU(inplace=True)

    def _low_pass_mask(self, h, w, device):
        cy, cx = h // 2, w // 2
        y = torch.arange(h, device=device).view(-1, 1).float() - cy
        x = torch.arange(w, device=device).view(1, -1).float() - cx
        dist = torch.sqrt(y ** 2 + x ** 2)
        radius = self.radius_ratio * min(h, w)
        return (dist <= radius).float()

    def forward(self, x):
        residual = x
        B, C, H, W = x.shape
        fft = torch.fft.fftshift(torch.fft.fft2(x, norm="ortho"), dim=(-2, -1))
        mask = self._low_pass_mask(H, W, x.device)[None, None]
        low = torch.fft.ifft2(torch.fft.ifftshift(fft * mask, dim=(-2, -1)), norm="ortho").real
        high = torch.fft.ifft2(torch.fft.ifftshift(fft * (1 - mask), dim=(-2, -1)), norm="ortho").real
        low = self.low_branch(low)
        high = self.high_branch(high)
        fused = self.fuse(torch.cat([low, high], dim=1))
        return self.act(fused + residual)


# ---------------------------------------------------------------------
# 4.4 Student Decoder (Reverse Distillation) with FDFM Stage 1 & 2
# ---------------------------------------------------------------------
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_fdfm=True):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.refine = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.fdfm = FDFM(out_ch) if use_fdfm else nn.Identity()

    def forward(self, x):
        x = self.up(x)
        x = self.refine(x)
        return self.fdfm(x)


class StudentDecoder(nn.Module):
    def __init__(self, bottleneck_channels=2048):
        super().__init__()
        self.up3 = DecoderBlock(bottleneck_channels, 1024, use_fdfm=True)  # FDFM Stage 1 -> reconstructs F3
        self.up2 = DecoderBlock(1024, 512, use_fdfm=True)                   # FDFM Stage 2 -> reconstructs F2
        self.up1 = DecoderBlock(512, 256, use_fdfm=False)                   # reconstructs F1

    def forward(self, bottleneck):
        d3 = self.up3(bottleneck)
        d2 = self.up2(d3)
        d1 = self.up1(d2)
        return [d1, d2, d3]   # matches [F1, F2, F3]


# ---------------------------------------------------------------------
# 4.5 Full FAD-RNet
# ---------------------------------------------------------------------
class FADRNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.teacher = TeacherEncoder(pretrained=pretrained)
        self.ocbe = OCBE(in_channels=(256, 512, 1024, 2048), out_channels=2048)
        self.decoder = StudentDecoder(bottleneck_channels=2048)

    def forward(self, x):
        with torch.no_grad():
            f1, f2, f3, f4 = self.teacher(x)
        bottleneck = self.ocbe([f1, f2, f3, f4])
        d1, d2, d3 = self.decoder(bottleneck)
        return [f1, f2, f3], [d1, d2, d3]


model = FADRNet(pretrained=True).to(DEVICE)
n_trainable = sum(p.numel() for p in model.ocbe.parameters() if p.requires_grad) + \
              sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in model.teacher.parameters())
print(f"Trainable params (OCBE + Student Decoder): {n_trainable/1e6:.2f} M")
print(f"Frozen teacher params: {n_frozen/1e6:.2f} M")


## 5. Loss function & anomaly map

In [ ]:

def cosine_loss(teacher_feats, student_feats):
    """1 - cosine similarity per spatial location, averaged over scales & pixels."""
    total = 0.0
    for t, s in zip(teacher_feats, student_feats):
        t_flat = t.view(t.shape[0], t.shape[1], -1)
        s_flat = s.view(s.shape[0], s.shape[1], -1)
        cos = F.cosine_similarity(t_flat, s_flat, dim=1)
        total = total + (1 - cos).mean()
    return total / len(teacher_feats)


def compute_anomaly_maps(teacher_feats, student_feats, out_size=256):
    """Per-scale (1-cos) map upsampled to out_size and averaged across scales.
    Returns a torch tensor (B, out_size, out_size) BEFORE Gaussian smoothing."""
    maps = []
    for t, s in zip(teacher_feats, student_feats):
        cos = F.cosine_similarity(t, s, dim=1)          # B,H,W
        amap = (1 - cos).unsqueeze(1)                     # B,1,H,W
        amap = F.interpolate(amap, size=(out_size, out_size),
                              mode="bilinear", align_corners=False)
        maps.append(amap)
    fused = torch.stack(maps, dim=0).mean(dim=0).squeeze(1)   # B,out,out
    return fused


def smooth_maps(maps_tensor, sigma=GAUSS_SIGMA):
    """Gaussian-smooth each map in a (B,H,W) tensor. Returns numpy array."""
    arr = maps_tensor.detach().cpu().numpy()
    return np.stack([gaussian_filter(arr[i], sigma=sigma) for i in range(arr.shape[0])])



## 6. Training

Trains only `OCBE` + `StudentDecoder` (teacher stays frozen) to minimize the cosine
reconstruction loss on **normal** fabric images. We track train & validation loss every
epoch, and checkpoint the model with the lowest validation loss (validation = held-out
normal images, so this is a legitimate unsupervised early-stopping signal — no test labels
are used anywhere in this cell).


In [ ]:

optimizer = torch.optim.Adam(
    list(model.ocbe.parameters()) + list(model.decoder.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
best_ckpt_path = os.path.join(OUTPUT_DIR, "fadrnet_best.pt")

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- train ----
    model.decoder.train(); model.ocbe.train(); model.teacher.eval()
    running = 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        t_feats, s_feats = model(imgs)
        loss = cosine_loss(t_feats, s_feats)
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    train_loss = running / len(train_loader.dataset)

    # ---- validate ----
    model.eval()
    running = 0.0
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            t_feats, s_feats = model(imgs)
            loss = cosine_loss(t_feats, s_feats)
            running += loss.item() * imgs.size(0)
    val_loss = running / len(val_loader.dataset)

    scheduler.step()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "ocbe": model.ocbe.state_dict(),
            "decoder": model.decoder.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
        }, best_ckpt_path)

    if epoch == 1 or epoch % 5 == 0 or epoch == NUM_EPOCHS:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | train_loss {train_loss:.4f} | "
              f"val_loss {val_loss:.4f} | best {best_val_loss:.4f}")

print(f"\nBest checkpoint saved to {best_ckpt_path} (val_loss={best_val_loss:.4f})")


### 6.1 Training curves

In [ ]:

plt.figure(figsize=(8,5))
plt.plot(history["train_loss"], label="Train loss")
plt.plot(history["val_loss"], label="Val loss (held-out normal images)")
plt.xlabel("Epoch"); plt.ylabel("Cosine reconstruction loss")
plt.title("FAD-RNet training curve")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(OUTPUT_DIR, "training_curve.png"), dpi=150, bbox_inches="tight")
plt.show()



## 7. Load best checkpoint & run inference on the test set

Computes, for every test image: the smoothed anomaly (pixel) map and an image-level
anomaly score (max of the smoothed map — the standard choice in RD4AD / PaDiM-style methods;
`np.mean(top-k)` is a robust alternative, see the note below the results).


In [ ]:

ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.ocbe.load_state_dict(ckpt["ocbe"])
model.decoder.load_state_dict(ckpt["decoder"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

all_scores, all_labels, all_classes, all_paths, all_maps = [], [], [], [], []

with torch.no_grad():
    for imgs, labels, cnames, paths in tqdm(test_loader, desc="Inference on test set"):
        imgs = imgs.to(DEVICE)
        t_feats, s_feats = model(imgs)
        raw_maps = compute_anomaly_maps(t_feats, s_feats, out_size=IMG_SIZE)
        smoothed = smooth_maps(raw_maps, sigma=GAUSS_SIGMA)          # B,H,W numpy
        # image score = max of smoothed anomaly map (top-1 pixel).
        # >>> EDIT: for noisier data, replace with top-k mean, e.g.:
        # scores = np.array([np.sort(m.ravel())[-50:].mean() for m in smoothed])
        scores = smoothed.reshape(smoothed.shape[0], -1).max(axis=1)

        all_scores.extend(scores.tolist())
        all_labels.extend(labels.tolist())
        all_classes.extend(list(cnames))
        all_paths.extend(list(paths))
        all_maps.append(smoothed)

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)          # 0 = good, 1 = defect
all_maps = np.concatenate(all_maps, axis=0)
print(f"Scored {len(all_scores)} test images.")



## 8. Image-level evaluation (ROC / AUROC / PR / confusion matrix)

These use the folder names as image-level ground truth (`good` = 0, any defect type = 1).
This is legitimate supervision **for evaluation only** — it was never used during training.


In [ ]:

auroc = roc_auc_score(all_labels, all_scores)
fpr, tpr, roc_thresh = roc_curve(all_labels, all_scores)
precision, recall, pr_thresh = precision_recall_curve(all_labels, all_scores)
pr_auc = auc(recall, precision)

# best threshold by Youden's J statistic (maximizes tpr - fpr)
best_idx = np.argmax(tpr - fpr)
best_threshold = roc_thresh[best_idx]

preds = (all_scores >= best_threshold).astype(int)
f1 = f1_score(all_labels, preds)

print(f"Image-level AUROC : {auroc:.4f}")
print(f"PR-AUC             : {pr_auc:.4f}")
print(f"Best threshold      : {best_threshold:.4f}  (Youden's J)")
print(f"F1 @ best threshold : {f1:.4f}")
print()
print(classification_report(all_labels, preds, target_names=["good", "defect"]))


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18,5))

# ROC curve
axes[0].plot(fpr, tpr, label=f"AUROC = {auroc:.3f}", linewidth=2)
axes[0].plot([0,1],[0,1],"--", color="gray")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve (image-level)")
axes[0].legend(); axes[0].grid(alpha=0.3)

# PR curve
axes[1].plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}", linewidth=2, color="darkorange")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(); axes[1].grid(alpha=0.3)

# Confusion matrix
cm = confusion_matrix(all_labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["good", "defect"])
disp.plot(ax=axes[2], cmap="Blues", colorbar=False)
axes[2].set_title(f"Confusion Matrix (thr={best_threshold:.3f})")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "evaluation_curves.png"), dpi=150, bbox_inches="tight")
plt.show()


### 8.1 Score distribution by class

In [ ]:

plt.figure(figsize=(9,5))
classes_present = sorted(set(all_classes))
for c in classes_present:
    vals = all_scores[np.array(all_classes) == c]
    sns.kdeplot(vals, label=c, fill=True, alpha=0.3)
plt.axvline(best_threshold, color="red", linestyle="--", label=f"threshold={best_threshold:.3f}")
plt.xlabel("Anomaly score (max smoothed map value)")
plt.ylabel("Density")
plt.title("Anomaly score distribution per class")
plt.legend()
plt.savefig(os.path.join(OUTPUT_DIR, "score_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()



## 9. Qualitative localization — anomaly maps overlaid on images

Since there are no ground-truth masks, these overlays are the way to *visually* confirm the
model is highlighting holes/lines/stains correctly. We show a few examples per class.


In [ ]:

def overlay_heatmap(img_tensor, amap, alpha=0.5):
    img = denorm(img_tensor).permute(1,2,0).numpy()
    amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
    heatmap = plt.cm.jet(amap_norm)[..., :3]
    return (1 - alpha) * img + alpha * heatmap

samples_per_class = 3
fig, axes = plt.subplots(len(classes_present), samples_per_class * 2,
                          figsize=(4 * samples_per_class * 2, 4 * len(classes_present)))
if len(classes_present) == 1:
    axes = axes[None, :]

for row, cname in enumerate(classes_present):
    idxs = [i for i, c in enumerate(all_classes) if c == cname][:samples_per_class]
    for col, idx in enumerate(idxs):
        img_tensor, _, _, path = test_ds[idx]
        amap = all_maps[idx]
        axes[row, col*2].imshow(denorm(img_tensor).permute(1,2,0).numpy())
        axes[row, col*2].set_title(f"{cname}\n(orig)"); axes[row, col*2].axis("off")
        axes[row, col*2+1].imshow(overlay_heatmap(img_tensor, amap))
        axes[row, col*2+1].set_title(f"score={all_scores[idx]:.3f}"); axes[row, col*2+1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "localization_examples.png"), dpi=150, bbox_inches="tight")
plt.show()


## 10. Save results, model, and metrics summary

In [ ]:

results_summary = {
    "auroc": float(auroc),
    "pr_auc": float(pr_auc),
    "best_threshold": float(best_threshold),
    "f1_at_best_threshold": float(f1),
    "num_test_images": int(len(all_scores)),
    "class_breakdown": {c: int((np.array(all_classes) == c).sum()) for c in classes_present},
    "best_epoch": int(ckpt["epoch"]),
    "best_val_loss": float(ckpt["val_loss"]),
    "config": {
        "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS,
        "lr": LR, "weight_decay": WEIGHT_DECAY, "gauss_sigma": GAUSS_SIGMA,
    },
}
with open(os.path.join(OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=2)

print(json.dumps(results_summary, indent=2))
print(f"\nAll artifacts saved under: {OUTPUT_DIR}")
print(" - fadrnet_best.pt           (model weights: OCBE + student decoder)")
print(" - training_curve.png")
print(" - evaluation_curves.png     (ROC / PR / confusion matrix)")
print(" - score_distribution.png")
print(" - localization_examples.png")
print(" - results_summary.json")


## 11. Single-image inference function (for deployment)

In [ ]:

def predict_single_image(image_path, threshold=best_threshold):
    img = Image.open(image_path).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        t_feats, s_feats = model(x)
        raw_map = compute_anomaly_maps(t_feats, s_feats, out_size=IMG_SIZE)
        smoothed = smooth_maps(raw_map, sigma=GAUSS_SIGMA)[0]
    score = smoothed.max()
    label = "DEFECT" if score >= threshold else "GOOD"

    fig, axes = plt.subplots(1, 2, figsize=(8,4))
    axes[0].imshow(img.resize((IMG_SIZE, IMG_SIZE))); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(overlay_heatmap(eval_transform(img), smoothed))
    axes[1].set_title(f"{label}  (score={score:.3f}, thr={threshold:.3f})"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
    return label, score, smoothed

# example usage (uncomment and point to any test image):
# predict_single_image(test_ds.samples[0][0])



## 12. What to change / modify — summary

| What | Where | Why |
|---|---|---|
| `DATA_ROOT` | Cell in **Section 2** | Must point at the folder containing `train/` and `test/`. Auto-detect usually finds it, but always check the printed path. |
| `NORMAL_CLASS_NAME` | Section 2 | Change if your "good" folder is named something else (e.g. `OK`, `normal`). |
| `IMG_SIZE`, `BATCH_SIZE`, `NUM_EPOCHS`, `LR` | Section 2 | Tune for your GPU / dataset size. Start with defaults; if `val_loss` is still dropping at epoch 100, increase `NUM_EPOCHS`. If you hit CUDA OOM, halve `BATCH_SIZE`. |
| `GAUSS_SIGMA` | Section 2 | Larger = smoother/blurrier anomaly maps (good for noisy backgrounds), smaller = sharper localization (good for fine defects like thin lines). Try 2–8. |
| Image score aggregation | Section 7, the `scores = ...` line | `max()` is sensitive to a single bad pixel; switch to the commented-out top-k mean if your maps are noisy and you get false positives on `good` images. |
| Backbone | `TeacherEncoder.__init__` in Section 4 | Swap `wide_resnet50_2` for `resnet50`/`resnext50_32x4d` from `torchvision.models` if you want a lighter/faster teacher — just update the channel tuple `(256,512,1024,2048)` passed to `OCBE` if the backbone's channel widths differ. |
| FDFM frequency split | `FDFM.__init__(radius_ratio=0.25)` in Section 4 | Fraction of the frequency spectrum treated as "low". Lower `radius_ratio` → more of the spectrum is treated as high-frequency (sharper defect emphasis). |
| Anomaly-map fusion | `compute_anomaly_maps` in Section 5 | Currently a simple mean across the 3 scales. You can weight scales differently, e.g. weight the finest scale (`F1`) higher for small defects. |
| Threshold | Section 8, `best_threshold` | Chosen by Youden's J on the ROC curve using test labels — good for reporting metrics. For a deployed system with *no* test labels available, instead pick threshold as a percentile (e.g. 95th/99th) of anomaly scores over the **validation (normal)** set — swap in: `threshold = np.percentile(val_scores, 99)`. |

### If you later obtain pixel-level ground-truth masks
Add a `mask` return to `LabeledTestDataset.__getitem__` (load the mask image the same way as
the input image, resized with `transforms.Resize((IMG_SIZE,IMG_SIZE), interpolation=NEAREST)`),
then compute pixel-level AUROC with:
```python
from sklearn.metrics import roc_auc_score
pixel_auroc = roc_auc_score(gt_masks.ravel(), all_maps.ravel())
```
and the PRO score using the standard `anomalib`/`RD4AD` reference implementation.
